# Analysis: the tables of one encoder from its scores

Run after `notebooks/04_train.ipynb` has finished for the encoder, at a fixed
commit, on the same kind of Colab runtime as the earlier notebooks (T4). The
analysis itself runs on the CPU. The encoder is set in the cell "What this
run does". The scores and the features are only read.

**`ENCODER = "hubert-base"` (as saved): the analysis of the plan and the
replication rule.** The tables of `docs/analysis_plan_hubert.md` (commit
`1ecf732`) are computed for HuBERT Base with the functions, resamples and
seeds that computed them for WavLM Base, and written under
`tables/hubert-base/`. Then:

- the units of HuBERT Base (`replication_units.csv`), listed by the function
  that listed the units of WavLM Base;
- the paired differences WavLM Base minus HuBERT Base, in the same resamples;
- the replication rule: for each of the 830 units of
  `docs/replication_units.csv` the verdict (`replication_comparison.csv`;
  the rule is applied to the 608 units in which the interval of WavLM Base
  excludes zero, every other unit is descriptive), the 2x2 outcome per row
  of the summary with the reverse count (`replication_outcome.csv`), the
  outcome per finding of the plan (`replication_findings.csv`) and the units
  of the absence findings (`replication_absence_findings.csv`).

Before any of this, the sha256 of the plan, of the units file and of its
summary are compared with `docs/preregistration.md`, and the units file of
WavLM Base on Drive with the committed one; a difference stops the notebook.
There are no original tables for HuBERT Base, so nothing is compared with an
original table.

A table that the plan does not name is written to
`tables/hubert-base/exploratory/`, with `exploratory_` at the start of its
file name, and its cell says EXPLORATORY. The cell "The pre-registered plan
and units" lists both kinds before anything is computed, and the last cell
stops if a table of the plan is missing or a table that is not listed was
written.

**`ENCODER = "wavlm-base"`: gate A3 and the units of the rule.** The tables
of WavLM Base are computed from the score files and compared with the
original tables. The original tables are read from their migrated copies
under `migration/original/tables/`, whose sha256 were compared with the
paper archive by `notebooks/02_migrate_wavlm_features.ipynb`. A table passes
if it has the same rows as the original table and every compared column
differs by at most the tolerance of its specification in the mapping file
(1e-9 unless the specification says otherwise); text columns must be equal.
The tables are written under `tables/wavlm-base/` and the check tables under
`checks/`.

1. inputs: the finished runs, speakers and attacks from the file lists, the original durations of In-the-Wild, the ASV error rates;
2. metrics per run and seed summary; the four train/test cells and the padded comparison;
3. fallback counts of the trimmed features;
4. the selected configurations and In-the-Wild by duration;
5. the effects of the 2x2 design (the longest step);
6. the rank agreement and the threshold summary;
7. the per-attack effect;
8. summary (hard stop if any row of any comparison fails);
9. the units of the replication rule, from the tables that passed the summary;
10. the noise ceiling of the rank agreement (`rank_noise_ceiling.csv`), not a table of the analysis plan.

For WavLM Base no comparison stops the notebook before the summary: every
check table is written first, and each cell prints whether its tables are
equal. The 20 original tables are compared in this order, the quick ones
first. Gate A3 passed at commit `88406e2`. The steps written at that commit
are read back (`REUSE_FROM` in the inputs cell), each checked against the
sha256 of its meta file and against the same inputs.

**The written tables of WavLM Base are not overwritten.** Gate A3, the
units of the replication rule and the numbers of the manuscript rest on
them. While `tables/wavlm-base/replication_units.csv` exists (the last table
of a full run), a run with `ENCODER = "wavlm-base"` does steps 1 and 10
only: it reads the inputs and computes the noise ceiling. No other table is
computed, compared or written, and no step is read back. The sha256 of every
file under `tables/wavlm-base/` and `checks/` is recorded before the inputs
are read, and the last cell of the noise ceiling stops unless every one of
them is unchanged and no file was added except those of the noise ceiling
(`analysis_files_unchanged_wavlm-base`). To run gate A3 again the tables of
WavLM Base have to be moved away first, by hand.

The bootstrap steps are long. On a laptop one train/test cell of the
evaluation set (1,000 resamples, five seeds) takes about 5 s. The steps of
HuBERT Base were not timed on Colab: it has all four cells at every layer
and every threshold (WavLM Base: at every layer at 40 dB, at layers 3 to 5
at 20 and 15 dB), so its effects take about twice as long as those of WavLM
Base, and the cells of WavLM Base are computed once more for the paired
differences. Every step is written to Drive when it is finished
(`tables/<encoder>/parts/`). After a disconnect the notebook is run again
from the top at the same commit: a finished step is read back instead of
computed again, as long as `runs.csv` has not changed.

In [ ]:
from google.colab import drive, userdata
import os

drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "Google Drive is not mounted"

### Repository at a fixed commit

Set `COMMIT` to the commit of this repository that is being checked: the
full hash or at least its first 7 characters. The repository is public and
is cloned without a token. If the Colab secret `GITHUB_TOKEN` exists and
this notebook has access to it, the clone uses it (a repository that is not
public needs one); the token is not kept in the clone and is replaced by
`***` in everything git prints.

In [ ]:
import subprocess

REPOSITORY = "Nooralabd/silence-shortcut-wavlm-layers"
COMMIT = ""          # the commit to run: the full hash or at least its first 7 characters
WORKDIR = "/content/silence-shortcut-wavlm-layers"

assert len(COMMIT) >= 7 and set(COMMIT) <= set("0123456789abcdef"), \
    "set COMMIT to the commit that is being checked (full hash or at least its first 7 characters)"
try:
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:    # the secret does not exist, or this notebook has no access to it
    token = ""       # a public repository is cloned without a token


def git(*arguments):
    """Run git and print what it prints; the token is never printed."""
    result = subprocess.run(["git", *arguments], capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if token:
        output = output.replace(token, "***")
    if output:
        print(output)
    assert result.returncode == 0, "git failed: see the lines above"
    return output


def clone(repository, folder):
    """Clone a repository, or fetch into an existing clone; a token, if there is one, is not kept in it."""
    address = f"https://{token}@github.com/{repository}.git" if token else f"https://github.com/{repository}.git"
    if not os.path.isdir(folder):
        git("clone", "-q", address, folder)
        git("-C", folder, "remote", "set-url", "origin", f"https://github.com/{repository}.git")
    else:
        git("-C", folder, "fetch", "-q", address, "+refs/heads/*:refs/remotes/origin/*")


clone(REPOSITORY, WORKDIR)
git("-C", WORKDIR, "checkout", "-q", COMMIT)
git("-C", WORKDIR, "rev-parse", "HEAD")
%cd {WORKDIR}

In [ ]:
# Installation. torch, torchaudio and torchcodec are the builds that extracted
# the original features: the pinned versions for CUDA 13.0. If anything pinned
# is missing or has another version, everything is installed in one pass and
# the cell always stops for a restart: a kernel never continues after pip has
# run in it (the runtime has imported numpy before this cell, and a module
# that is already loaded stays loaded when pip replaces it on disk).
# On the run after the restart nothing is installed: the cell checks the
# installed versions and the versions of the modules that the kernel loads.
import importlib
import importlib.metadata
import re
import sys

TORCH_INDEX = "https://download.pytorch.org/whl/cu130"
TORCH_FAMILY = ("torch", "torchaudio", "torchcodec")          # installed from TORCH_INDEX
INSTALLED_ONCE = os.path.join(os.path.dirname(WORKDIR), "installed_once.txt")
OTHERS = os.path.join(os.path.dirname(WORKDIR), "requirements_without_torch.txt")
CONSTRAINT = os.path.join(os.path.dirname(WORKDIR), "torch_constraint.txt")

with open(os.path.join(WORKDIR, "requirements.txt"), encoding="utf-8") as fh:
    requirements = [line.strip() for line in fh if line.strip() and not line.lstrip().startswith("#")]
pins = dict(line.split("==") for line in requirements if "==" in line)
BUILDS = {name: pins[name] + "+" + TORCH_INDEX.rsplit("/", 1)[1] for name in TORCH_FAMILY}


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def not_as_pinned():
    """{package: (installed, wanted)} for everything that is missing or has another version."""
    found = {}
    for line in requirements:
        name = re.split(r"[=<>!~ ]", line, maxsplit=1)[0]
        have, want = installed(name), BUILDS.get(name, pins.get(name))
        same = have is not None and (want is None or (have if name in BUILDS else have.split("+")[0]) == want)
        if not same:
            found[name] = (have, want)
    if installed("ssl-silence") is None:
        found["ssl-silence"] = (None, "this repository")
    return found


different = not_as_pinned()
if different:
    assert not os.path.exists(INSTALLED_ONCE), (
        f"the libraries were installed once already and these are still not as pinned: {different}. "
        "Stop: do not restart again")
    imported = sorted(name for name in (*TORCH_FAMILY, "transformers", "ssl_silence") if name in sys.modules)
    assert not imported, (f"{imported} were imported before the installation: "
                          "Runtime > Restart session, then run the notebook again from the top")
    with open(OTHERS, "w", encoding="utf-8") as fh:
        for line in requirements:
            if re.split(r"[=<>!~ ]", line, maxsplit=1)[0] not in TORCH_FAMILY:
                print(line, file=fh)
    with open(CONSTRAINT, "w", encoding="utf-8") as fh:
        for name in TORCH_FAMILY:
            print(f"{name}=={BUILDS[name]}", file=fh)
    # The build is part of the pin (torch==2.11.0+cu130): for a pin without it pip
    # keeps another build of the same version that the runtime ships (+cpu).
    FAMILY_PINS = " ".join(f"{name}=={BUILDS[name]}" for name in TORCH_FAMILY)
    # torchvision of the runtime is built for its own torch and breaks imports
    # with another one; this package does not use it.
    !pip uninstall -q -y torchvision
    !pip install -q {FAMILY_PINS} --index-url {TORCH_INDEX}
    !pip install -q -r {OTHERS} -c {CONSTRAINT}
    !pip install -q --no-deps -e .
    importlib.invalidate_caches()
    # Before the restart: everything on disk is as pinned, so the next run installs nothing.
    different = not_as_pinned()
    assert not different, f"pip did not install these as pinned: {different}: stop"
    with open(INSTALLED_ONCE, "w", encoding="utf-8") as fh:
        print("the pinned libraries were installed in this runtime", file=fh)
    # pip has run in this kernel: it always stops here.
    raise RuntimeError("The pinned libraries were installed. "
                       "Runtime > Restart session, then run the notebook again from the top")

# Nothing was installed in this run. The modules that this kernel loads must
# be the pinned versions too, not only the files on disk.
LOADED = {"numpy": "numpy", "scipy": "scipy", "pandas": "pandas", "scikit-learn": "sklearn",
          "torch": "torch", "torchaudio": "torchaudio", "transformers": "transformers"}
for name, module in LOADED.items():
    loaded = importlib.import_module(module).__version__
    assert loaded.split("+")[0] == pins[name], (
        f"{module} {loaded} is loaded in this kernel and the pin is {pins[name]}: it was imported before pip "
        "replaced it. Runtime > Restart session, then run the notebook again from the top")

import torch
import torchaudio
import transformers

print("loaded:", {name: sys.modules[module].__version__ for name, module in LOADED.items()})
print("torch", torch.__version__, "| CUDA of the build:", torch.version.cuda)
print("torchaudio", torchaudio.__version__, "| torchcodec", installed("torchcodec"),
      "| transformers", transformers.__version__)
if os.path.join(WORKDIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(WORKDIR, "src"))   # a running kernel does not see a new editable install

In [ ]:
import subprocess
import sys

import pandas as pd

from ssl_silence import config as cfg
from ssl_silence import provenance, verify

pd.set_option("display.max_rows", 200, "display.width", 200)
HEAD = provenance.git_commit()     # the full hash, with "-dirty" if a tracked file was changed
assert HEAD.startswith(COMMIT) and not HEAD.endswith("-dirty"), (HEAD, COMMIT)
print("commit:", HEAD)
print("root:", cfg.ROOT)

In [ ]:
# The quota of the Google account is read from the Drive API (this asks for
# access to the account); the mounted folder does not report it.
QUOTA = cfg.drive_quota()
print("Google Drive: {usage_gb:.1f} GB used of {limit_gb:.1f} GB, {free_gb:.1f} GB free".format(**QUOTA),
      "(GB = 10^9 bytes; required: %d GB free)" % cfg.MIN_FREE_GB)
cfg.require_free_space(quota=QUOTA)
print("local disk of the runtime: %.1f GB free" % cfg.local_free_space_gb())

### Mapping file

`legacy_map.yaml` names the original folders, files and labels. It is kept on
Drive only, at `<root>/migration/legacy_map.yaml`, and must have no empty
value (`verify.load_map` stops otherwise).

In [ ]:
MAP_PATH = os.path.join(cfg.migration_dir(), "legacy_map.yaml")
assert os.path.exists(MAP_PATH), f"upload legacy_map.yaml to {MAP_PATH}"
legacy = verify.load_map(MAP_PATH)
FORBIDDEN = legacy["forbidden"]
CHECKS = cfg.checks_dir()
checks = {}


def record(name, table, stop=True):
    """Write a check table to Drive, show it, and stop if a row fails."""
    path = verify.write_check(table, name, CHECKS, FORBIDDEN)
    checks[name] = table
    display(table)
    if stop and "ok" in table.columns:
        verify.require_ok(table, name)
    return path

### What this run does

`ENCODER` is the only value to set. For WavLM Base the run is the noise
ceiling only if its tables exist (see the top of the notebook).

In [ ]:
from ssl_silence import analysis, replication

ENCODER = "hubert-base"    # "hubert-base": the analysis of the plan and the replication rule; "wavlm-base": gate A3

REFERENCE = "wavlm-base"                 # the encoder of the original analysis
REPLICATION = ENCODER != REFERENCE       # original tables exist for the reference encoder only
assert ENCODER in cfg.experiments()["experiments"], \
    f"ENCODER is {ENCODER!r}: set it to one of {list(cfg.experiments()['experiments'])}"
KINDS = [kind for kind in cfg.EXTRACTIONS if kind in cfg.experiments()["experiments"][ENCODER]]

# The tables of WavLM Base are never computed again and never written while they exist: gate A3, the
# units of the replication rule and the manuscript rest on them. replication_units.csv is the last
# table of a full run.
CEILING_ONLY = not REPLICATION and os.path.exists(os.path.join(cfg.tables_dir(REFERENCE), "replication_units.csv"))
TABLES = not CEILING_ONLY                # this run computes the tables of the encoder
STEPS = (f"analysis_steps_{ENCODER}" if REPLICATION else "analysis_steps_noise_ceiling" if CEILING_ONLY
         else "analysis_steps")          # name of the check table of the steps
UNCHANGED = f"analysis_files_unchanged_{ENCODER}"
GUARDED = [cfg.tables_dir(REFERENCE), CHECKS]
OWN = ("rank_noise_ceiling", STEPS, UNCHANGED)       # the start of the name of every file this run writes
BEFORE = verify.file_hashes(GUARDED, skip=OWN) if CEILING_ONLY else {}
print("this run:", ENCODER, "| extractions:", KINDS, "|",
      f"the tables of the analysis plan and the replication rule against {REFERENCE}" if REPLICATION
      else "the noise ceiling of the rank agreement only; no other table is computed or written" if CEILING_ONLY
      else "gate A3 (comparison with the original tables) and the units of the replication rule")
if CEILING_ONLY:
    print("files of", REFERENCE, "and check tables that exist:", len(BEFORE),
          "| their sha256 is recorded and must not change")

### The pre-registered plan and units

For the replication only, and before any table of the second encoder is
computed: the sha256 of `docs/analysis_plan_hubert.md`,
`docs/replication_units.csv` and `docs/replication_units_summary.csv` of the
checked-out commit must equal those recorded in `docs/preregistration.md`,
and the units file of WavLM Base on Drive must equal the committed one. A
difference stops the notebook here.

The table below lists every table this run writes: the tables of the
analysis plan with the section that names them, the tables written only if
a dev-selected layer lies outside layers 3 to 5, and the EXPLORATORY tables
(not in the plan; separate folder, `exploratory_` in the file name).

In [ ]:
REFERENCE_UNITS = os.path.join(WORKDIR, replication.PREREGISTERED_UNITS)
if REPLICATION:
    record(f"preregistration_{ENCODER}", replication.preregistration_check(WORKDIR, {
        os.path.join(cfg.tables_dir(REFERENCE), "replication_units.csv"): replication.PREREGISTERED_UNITS}))
    reference_units = replication.read_units(REFERENCE_UNITS)
    registered = analysis.read_table(os.path.join(WORKDIR, replication.PREREGISTERED_SUMMARY))
    counted = replication.summarise_units(reference_units)
    for column in ("n_units", "n_in_rule", "n_positive_in_rule", "n_negative_in_rule"):
        assert counted[column].tolist() == registered[column].tolist(), \
            f"the units file does not give the {column} of its pre-registered summary"
    print("the pre-registered files are unchanged | units of", REFERENCE + ":", len(reference_units),
          "| in the rule:", int(reference_units["in_rule"].sum()))
    with pd.option_context("display.max_colwidth", None):
        display(analysis.planned_tables())
else:
    print("not a replication: the pre-registered files are not read")

### Inputs

The speakers and the attacks come from the file lists of the datasets (the
archives are unpacked to the local disk for their protocol and metadata
files; this takes a few minutes). The original duration of every
In-the-Wild file is read from the migrated copy of the original per-file
measurement table, because `notebooks/05_silence_measurements.ipynb` has not
measured the files again yet; the number of files on each side of 4 s is
checked against `config.ITW_DURATION_COUNTS`. The durations do not depend on
the encoder.

For the replication the scores of the reference encoder are read too, and
the resamples of the two encoders are compared: the same files in the same
order, the same labels and speakers, and the same multiplicities in every
file-level and speaker-level resample. A difference stops the notebook.

In [ ]:
from ssl_silence import datasets, selection, silence, training

ORIGINAL = os.path.join(cfg.migration_dir(), "original")
RUNS = {kind: cfg.runs_dir(ENCODER, kind) for kind in KINDS}
for kind in KINDS:
    planned = [run.spec.run_id for run in training.plan_runs(ENCODER, kind)]
    missing = sorted(set(planned) - training.finished_run_ids(RUNS[kind]))
    assert not missing, f"{len(missing)} {kind} runs are not finished: run notebooks/04_train.ipynb first"
    print(kind, "| finished runs:", len(planned), "|", RUNS[kind])
if REPLICATION:
    planned = [run.spec.run_id for run in training.plan_runs(REFERENCE, "unpadded")]
    missing = sorted(set(planned) - training.finished_run_ids(cfg.runs_dir(REFERENCE, "unpadded")))
    assert not missing, f"{len(missing)} unpadded runs of {REFERENCE} are not finished"

datasets.ensure_asvspoof(cfg.la_zip())
datasets.ensure_itw(cfg.itw_zip(), sha256=cfg.ITW_ZIP_SHA256)
file_lists = {name: datasets.items(name) for name in cfg.TEST_SETS}
for name, file_list in file_lists.items():
    print(name, datasets.check_items(file_list, name))
durations = silence.duration_map(silence.read_measurements(
    os.path.join(ORIGINAL, "files", "measurements_dir", "itw.csv")))
print("In-the-Wild files of at most 4 s, longer:", silence.check_duration_subsets(durations))
asv = selection.load_asv(cfg.asv_scores_dir())
assert set(asv) == {"dev", "eval"}, "the official ASV score files are missing: run notebooks/02_migrate_wavlm_features.ipynb"

# Steps written at these commits are read back as well (full hashes): the commit at which gate A3 passed.
REUSE_FROM = ("88406e2d1f35ae39a51e033262c7e7db99774465",)
FILES = dict(speaker_maps={name: datasets.speaker_map(file_lists[name]) for name in cfg.TEST_SETS},
             attack_map=datasets.attack_map(file_lists["eval"]), duration_map=durations, asv=asv)
tables = analysis.ScoreTables(
    ENCODER, RUNS["unpadded"], cfg.tables_dir(ENCODER), reuse_commits=() if REPLICATION else REUSE_FROM,
    padded_runs_dir=RUNS.get("padded"), **FILES)
EXPECTED = analysis.original_table_names() if TABLES and not REPLICATION else {}
print("trimmed conditions:", tables.conditions, "| layers with all four cells:", tables.cross_layers)
print("resamples:", tables.n_boot, "| seeds:", tables.seeds)
print("original tables to compare:", sum(len(names) for names in EXPECTED.values()))
if REPLICATION:
    # The reference encoder: its scores are read for the paired differences; none of its tables is written.
    reference = analysis.ScoreTables(REFERENCE, cfg.runs_dir(REFERENCE, "unpadded"), cfg.tables_dir(REFERENCE),
                                     **FILES)
    assert (reference.n_boot, reference.seeds) == (tables.n_boot, tables.seeds)
    record(f"resamples_{ENCODER}_against_{REFERENCE}", analysis.paired_resamples_check(reference, tables))
    print("the resamples of", ENCODER, "are those of", REFERENCE)

In [ ]:
def compare(name, table):
    """A table of this package against its original tables. Every check table
    is written; a difference is printed here and stops the notebook in the summary.
    There are no original tables for a second encoder: its table is only written."""
    if REPLICATION:
        print(name, "| written |", len(table), "rows")
        return
    for check_name, check in analysis.compare_with_original(name, table, legacy, ORIGINAL).items():
        record(check_name, check, stop=False)
        print(check_name, "| equal" if check["ok"].all() else "| DIFFERENT: see the table above")

### Metrics per run, seed summary, cells and the padded comparison

`metrics_per_run.csv` (official EER of every score set, tandem t-DCF of the
dev and evaluation sets) and `seed_summary.csv` (mean and sample SD over
seeds). The cells of 40 dB and the padded comparison are the first two
comparisons of WavLM Base. HuBERT Base has no padded runs.

In [ ]:
if TABLES:
    summary = tables.metrics()
    print("cells in the seed summary:", len(summary), "| seeds per cell:", sorted(set(summary["n_seeds"])))
    compare("cells_2x2", tables.cells())
    if "padded" in KINDS:
        compare("padded_vs_unpadded", tables.padded_vs_unpadded())

### Fallback counts

Files that kept their untrimmed audio, per split and class, read from the
shards of the trimmed features on Drive.

In [ ]:
if TABLES:
    compare("fallback_counts", tables.fallback_counts(
        {condition: cfg.feature_dir(ENCODER, "unpadded", condition) for condition in tables.conditions}))

### Selected configurations and In-the-Wild by duration

The selected configurations with their file- and speaker-level intervals
(the first use of the resamples of both test sets), and the cells and
effects on In-the-Wild by original duration.

WavLM Base: the reference and the dev-selected configurations of the
untrimmed and the 40 dB condition, and the duration table as the original
analysis wrote it. HuBERT Base has no reference configuration: the
dev-selected layer of every condition and pooling (`dev_selected.csv`, with
intervals in `selected_configs_eer_ci.csv`), and the duration table at
40 dB, which is what the plan names.

**EXPLORATORY (HuBERT Base only, not in the analysis plan):**
`exploratory/exploratory_itw_by_duration_thresholds.csv`, the same duration
table at 20 and 15 dB. No interval, no unit of the rule and no finding
depends on it.

In [ ]:
if TABLES:
    compare("selected_configs_eer_ci", tables.selected_configs_eer_ci(all_conditions=REPLICATION))
    if REPLICATION:
        display(tables.dev_selected())
        compare("itw_by_duration", tables.itw_by_duration((replication.MAIN_CONDITION,)))
        exploratory = tables.itw_by_duration_thresholds()      # EXPLORATORY: not in the analysis plan
        print("EXPLORATORY, not in the analysis plan:", tables.exploratory_path("exploratory_itw_by_duration_thresholds"),
              "|", len(exploratory), "rows")
    else:
        compare("itw_by_duration", tables.itw_by_duration())

### Effects of the 2x2 design

The six effects with paired file- and speaker-level intervals, for the
layers with all four cells: for WavLM Base every layer at 40 dB and layers
3 to 5 at 20 and 15 dB, for HuBERT Base every layer at every threshold. This
is the longest step; each condition is written when it is finished.

In [ ]:
if TABLES:
    compare("effects_2x2", tables.effects())

### Rank agreement and threshold summary

Kendall's tau and Spearman's rho between the layer curves of the untrimmed
and of each trimmed condition, with percentile and BCa intervals, and the
share of resamples in which each layer is best; then the thresholds side by
side, from the tables written above.

In [ ]:
if TABLES:
    rank, rank_layers = tables.rank_agreement()
    compare("rank_agreement", rank)
    compare("rank_agreement_layers", rank_layers)
    compare("threshold_summary", tables.threshold_summary())

### Per-attack effect

In [ ]:
if TABLES:
    compare("per_attack_effect", tables.per_attack_effect())

### Summary

WavLM Base: the hard stop of gate A3. HuBERT Base: nothing was compared.

In [ ]:
if TABLES:
    record(STEPS, tables.steps_table(), stop=False)
    print("tables written to", cfg.tables_dir(ENCODER), "| check tables written to", CHECKS)
    if REPLICATION:
        print("no original tables for", ENCODER + ": the tables above were written, not compared")
    else:
        compared = {name: table for name, table in checks.items() if name.startswith("table_")}
        summary = verify.summarise_checks(compared)
        display(summary)
        expected = {f"table_{original}" for names in EXPECTED.values() for original in names}
        assert set(compared) == expected, f"not compared: {sorted(expected - set(compared))}"
        verify.require_ok(pd.concat(list(compared.values()), ignore_index=True), "analysis tables against the original tables")
        print("gate A3 passed:", ENCODER, "| original tables compared:", len(compared),
              "| columns compared:", int(summary["n_rows"].sum()))

### Units of the replication rule

For WavLM Base computed only after the summary above has passed. Three
tables are added:

- `effects_matched_cells.csv`: the total effect at 20 and 15 dB for every
  layer, from the two matched cells (for WavLM Base the cross-condition
  cells exist there for layers 3 to 5 only). Where the effects table has the
  row too, the two must be identical;
- `rank_agreement_threshold_difference.csv`: Kendall's tau at 40 dB minus tau
  at each stricter threshold, in the same resamples;
- `itw_by_duration_ci.csv` (and `itw_by_duration_difference.csv`): the six
  effects per duration subset at 40 dB with paired intervals.

`replication_units.csv` lists every unit with its estimate, its speaker-level
interval, its sign and `in_rule` (the interval excludes zero);
`replication_units_summary.csv` counts them per quantity, threshold, test
set and subset. The file of WavLM Base was copied by hand into `docs/` of
the repository and is pre-registered. The file of HuBERT Base has the same
units; its `in_rule` says whether the interval of HuBERT Base excludes zero.

In [ ]:
if TABLES:
    matched = tables.matched_total_effects()
    differences = tables.tau_differences()
    by_subset, subset_difference = tables.itw_by_duration_ci()
    units, units_summary = tables.replication_units()
    record(STEPS, tables.steps_table(), stop=False)
    display(units_summary)
    print("units:", len(units), "| in the rule:", int(units["in_rule"].sum()),
          "| by condition:", units.groupby("condition").size().to_dict())
    if not REPLICATION:
        print("copy", tables.path("replication_units"), "to docs/replication_units.csv of the repository")

### Noise ceiling of the rank agreement (WavLM Base only)

Not a table of the analysis plan, and not computed for HuBERT Base. It asks
how well two layer rankings agree when only the speakers differ, as a
reference for Kendall's tau between the clean and the trimmed ranking.

The speakers of each test set are split at random into two halves, within
each class, 1,000 times (the number of resamples; the bootstrap seed with
its own stream per test set and pooling). On each half the curve is the
seed-mean official EER per layer. In every split:

- the ceiling is tau between the clean-to-clean curves of the two halves;
- the cross-condition tau is tau between the clean-to-clean curve of one
  half and the trimmed-to-trimmed curve (40 dB) of the other half.

`rank_noise_ceiling.csv` has one row per test set and pooling: the tau of
all files (clean against trimmed), and the median with the 2.5th and 97.5th
percentiles of the ceiling and of the cross-condition tau, and the share of
splits whose ceiling is at or below the tau of all files.
`rank_noise_ceiling_splits.csv` has the two values of every split. Nothing
is retrained; the definition was fixed by the first author before this cell
was written (`docs/decisions.md`, Section 3).

If the tables of WavLM Base existed at the start, this is the only step of
the run, and the cell stops unless every file that existed then is
unchanged.

In [ ]:
if not REPLICATION:
    ceiling, ceiling_splits = tables.rank_noise_ceiling()
    record(STEPS, tables.steps_table(), stop=False)
    display(ceiling)
    print("noise ceiling:", len(ceiling), "rows |", len(ceiling_splits), "splits |",
          tables.path("rank_noise_ceiling"))
if CEILING_ONLY:
    record(UNCHANGED, verify.unchanged_check(BEFORE, GUARDED, skip=OWN))      # a hard stop
    print("noise ceiling run finished:", ENCODER, "| files that existed at the start:", len(BEFORE),
          "unchanged | no other table was computed or written")

### Dev-selected layers outside layers 3 to 5 (replication only)

The duration and per-attack analyses use layers 3 to 5. If a dev-selected
layer of HuBERT Base (any condition, either pooling) lies outside them, the
plan asks for the same analyses for that layer: `per_attack_effect`,
`itw_by_duration`, `itw_by_duration_ci` and `itw_by_duration_difference`,
each with `_dev_selected_layers` at the end of its name. They add no unit to
the rule. With no such layer nothing is written.

In [ ]:
if REPLICATION:
    outside = tables.selected_layers_outside_detail()
    additional = tables.selected_layer_tables()
    print("dev-selected layers outside", tables.detail_layers, ":", list(outside) or "none",
          "| tables:", list(additional) or "none")

### Paired differences between the encoders (replication only)

WavLM Base minus HuBERT Base, computed inside every resample, with file- and
speaker-level intervals: the four cells and the six effects for the layers
with all four cells in both encoders (every layer at 40 dB, layers 3 to 5 at
20 and 15 dB), the two matched cells and the total effect for the other
layers at 20 and 15 dB (`encoder_difference_cells_effects.csv`), and
Kendall's tau (`encoder_difference_rank_agreement.csv`). These differences
are descriptive; no verdict depends on them.

In [ ]:
if REPLICATION:
    encoder_effects = tables.encoder_differences(reference)
    encoder_tau = tables.encoder_tau_differences(reference)
    record(STEPS, tables.steps_table(), stop=False)
    print("paired differences:", len(encoder_effects), "rows of cells and effects |", len(encoder_tau), "rows of tau")

### The replication rule (replication only)

The rule of Section 6 of the plan, applied to the units of
`docs/replication_units.csv` and the units of HuBERT Base written above:

- a unit **in the rule** (the speaker-level percentile interval of WavLM
  Base excludes zero; 608 of the 830 units) is *replicated* if the estimate
  of HuBERT Base has the same sign and its speaker-level percentile interval
  excludes zero. Otherwise it is *not replicated*. Kendall's tau follows
  the same rule; `sign` and `replica_sign` in `replication_comparison.csv`
  are the signs of the two estimates of every unit;
- every other unit is *descriptive*: no verdict.

`replication_outcome.csv` has one row per row of the summary: the units, the
units in the rule, the replicated units and the 2x2 classification
(`both_exclude_same_sign`, `both_exclude_opposite_sign`, `reference_only`,
`replica_only`, `neither`). `replica_only` is the reverse count: the
interval of HuBERT Base excludes zero where that of WavLM Base did not.
`replication_findings.csv` gives the same counts per finding of Section 6a.
The absence findings (F7 and F16) are reported with the 2x2 classification
and their units (`replication_absence_findings.csv`); they are counted
neither as replicated nor as failed, so their `n_replicated` is empty. No
correction for multiple comparisons, and no unit is interpreted on its own.
The places where the code had to read the plan in one of two ways are in
`docs/deviations.md`.

In [ ]:
if REPLICATION:
    outcome = tables.replication_outcome(REFERENCE_UNITS)
    comparison = outcome["replication_comparison"]
    with pd.option_context("display.max_rows", 200):
        display(outcome["replication_outcome"])
        display(outcome["replication_findings"])
        display(outcome["replication_absence_findings"])
    assert len(comparison) == len(reference_units) and \
        int(comparison["in_rule"].sum()) == int(reference_units["in_rule"].sum())
    print("units:", len(comparison), "| in the rule:", int(comparison["in_rule"].sum()),
          "| verdicts:", comparison["verdict"].value_counts().to_dict(),
          "| reverse count:", int(outcome["replication_outcome"]["replica_only"].sum()))

### Tables written (replication only)

Every table of the analysis plan must exist, the tables for dev-selected
layers exactly when there is such a layer, the exploratory table in its own
folder, and no table that is not listed. A failing row stops the notebook.

In [ ]:
if REPLICATION:
    record(STEPS, tables.steps_table(), stop=False)
    written = tables.written_tables_check()
    record(f"analysis_tables_{ENCODER}", written)
    print("analysis finished:", ENCODER, "| tables of the analysis plan:",
          int((written["kind"].str.startswith("analysis plan") & written["written"]).sum()),
          "| exploratory tables:", int((written["kind"] == "exploratory").sum()),
          "| folder:", cfg.tables_dir(ENCODER))